In [1]:
import openfermion as of
import sympy
from itertools import product

In [2]:
a = sympy.Symbol("a")

op_signature = ((0, 1), (0, 0))

op = of.FermionOperator(op_signature, a)

In [3]:
h_11, h_22, h_12 = sympy.symbols("h_{11} h_{22} h_{12}")

h_11 = 0
h_22 = 0

In [122]:
g_1111, g_1112, g_1122, g_1212, g_1222, g_2222 = sympy.symbols(
    "g_{1111}, g_{1112}, g_{1122}, g_{1212}, g_{1222}, g_{2222}"
)
# g_1111 = 0
# g_1112 = 0
# g_1212 = 0
# g_1122 = 0
# g_1222 = 0
# g_2222 = 0

Interleaved spin order

In [123]:
number_terms = [((i, 1), (i, 0)) for i in range(4)]

cross_terms = [((i, 1), ((i + 2) % 4, 0)) for i in range(4)]

# print(cross_terms)

number_terms_coeffs = [h_11, h_11, h_22, h_22]

one_body_hamiltonian = of.FermionOperator()

for i in range(4):
    one_body_hamiltonian += of.FermionOperator(number_terms[i], number_terms_coeffs[i])
    one_body_hamiltonian += of.FermionOperator(cross_terms[i], h_12)

print(one_body_hamiltonian)

h_{12} [0^ 2] +
h_{12} [1^ 3] +
h_{12} [2^ 0] +
h_{12} [3^ 1]


In [124]:
# a_1, b_1, c_1 = sympy.symbols("a_1 b_1 c_1")
# a_2, b_2, c_2 = sympy.symbols("a_2 b_2 c_2")

a_1, b_1, c_1 = 1, 1, -1
a_2, b_2, c_2 = 1, 1, -2

symmetry_1 = (of.FermionOperator(number_terms[0], a_1)
              + of.FermionOperator(number_terms[1], b_1)
              + of.FermionOperator(number_terms[0], c_1) * of.FermionOperator(number_terms[1], 1))

print(symmetry_1)

symmetry_2 = (of.FermionOperator(number_terms[2], a_2)
              + of.FermionOperator(number_terms[3], b_2)
              + of.FermionOperator(number_terms[2], c_2) * of.FermionOperator(number_terms[3], 1))

1 [0^ 0] +
-1.0 [0^ 0 1^ 1] +
1.0 [1^ 1]


In [125]:
of.normal_ordered(of.commutator(one_body_hamiltonian, symmetry_1)).induced_norm()

8.0*Abs(h_{12})**1.0

In [126]:
def coeff_class(i, j, k, el):
    modified_coeffs = [i, j, k, el]
    if i > j:
        modified_coeffs[0], modified_coeffs[1] = modified_coeffs[1], modified_coeffs[0]
    if k > el:
        modified_coeffs[2], modified_coeffs[3] = modified_coeffs[3], modified_coeffs[2]
    if (10 * i + j) > (10 * k + el):
        modified_coeffs[0], modified_coeffs[2] = modified_coeffs[2], modified_coeffs[0]
        modified_coeffs[1], modified_coeffs[3] = modified_coeffs[3], modified_coeffs[1]
    modified_coeffs = tuple(modified_coeffs)
    if modified_coeffs == (0, 0, 0, 0):
        return g_1111
    elif modified_coeffs == (0, 0, 0, 1):
        return g_1112
    elif modified_coeffs == (0, 0, 1, 1):
        return g_1122
    elif modified_coeffs == (0, 1, 0, 1):
        return g_1212
    elif modified_coeffs == (0, 1, 1, 1):
        return g_1222
    elif modified_coeffs == (1, 1, 1, 1):
        return g_2222
    else:
        print(modified_coeffs)
        raise ValueError()

In [127]:
two_body_terms = [((2 * i + sigma, 1),(2 * k + tau, 1),(2 * el + tau, 0),(2 * j + sigma, 0))
                 for i, j, k, el, sigma, tau in product(range(2),repeat=6)]

two_body_coeffs = [coeff_class(i, j, k, el)
                  for i, j, k, el, sigma, tau in product(range(2),repeat=6)]

two_body_hamiltonian = of.FermionOperator()
for i, term in enumerate(two_body_terms):
    two_body_hamiltonian += of.FermionOperator(term, two_body_coeffs[i] / 2)

In [128]:
def pretty_printed_index(j: int):
    if j % 2 == 0:
        spin = "a"
    else:
        spin = "b"
    return str(j // 2 + 1) + spin

In [129]:
for k, v in two_body_hamiltonian.terms.items():
    # print(v)
    # if v == g_1212 / 2:
    print(v, [pretty_printed_index(w[0]) for w in k])

g_{1111}/2 ['1a', '1a', '1a', '1a']
g_{1111}/2 ['1a', '1b', '1b', '1a']
g_{1111}/2 ['1b', '1a', '1a', '1b']
g_{1111}/2 ['1b', '1b', '1b', '1b']
g_{1112}/2 ['1a', '1a', '2a', '1a']
g_{1112}/2 ['1a', '1b', '2b', '1a']
g_{1112}/2 ['1b', '1a', '2a', '1b']
g_{1112}/2 ['1b', '1b', '2b', '1b']
g_{1112}/2 ['1a', '2a', '1a', '1a']
g_{1112}/2 ['1a', '2b', '1b', '1a']
g_{1112}/2 ['1b', '2a', '1a', '1b']
g_{1112}/2 ['1b', '2b', '1b', '1b']
g_{1122}/2 ['1a', '2a', '2a', '1a']
g_{1122}/2 ['1a', '2b', '2b', '1a']
g_{1122}/2 ['1b', '2a', '2a', '1b']
g_{1122}/2 ['1b', '2b', '2b', '1b']
g_{1112}/2 ['1a', '1a', '1a', '2a']
g_{1112}/2 ['1a', '1b', '1b', '2a']
g_{1112}/2 ['1b', '1a', '1a', '2b']
g_{1112}/2 ['1b', '1b', '1b', '2b']
g_{1212}/2 ['1a', '1a', '2a', '2a']
g_{1212}/2 ['1a', '1b', '2b', '2a']
g_{1212}/2 ['1b', '1a', '2a', '2b']
g_{1212}/2 ['1b', '1b', '2b', '2b']
g_{1212}/2 ['1a', '2a', '1a', '2a']
g_{1212}/2 ['1a', '2b', '1b', '2a']
g_{1212}/2 ['1b', '2a', '1a', '2b']
g_{1212}/2 ['1b', '2b', '1b'

In [130]:
of.normal_ordered(two_body_hamiltonian)

-g_{1111} [1^ 0^ 1 0] +
g_{1112} [1^ 0^ 2 1] +
-g_{1112} [1^ 0^ 3 0] +
-g_{1212} [1^ 0^ 3 2] +
-g_{1122} + g_{1212} [2^ 0^ 2 0] +
g_{1112} [2^ 1^ 1 0] +
-g_{1122} [2^ 1^ 2 1] +
g_{1212} [2^ 1^ 3 0] +
g_{1222} [2^ 1^ 3 2] +
-g_{1112} [3^ 0^ 1 0] +
g_{1212} [3^ 0^ 2 1] +
-g_{1122} [3^ 0^ 3 0] +
-g_{1222} [3^ 0^ 3 2] +
-g_{1122} + g_{1212} [3^ 1^ 3 1] +
-g_{1212} [3^ 2^ 1 0] +
g_{1222} [3^ 2^ 2 1] +
-g_{1222} [3^ 2^ 3 0] +
-g_{2222} [3^ 2^ 3 2]

In [131]:
of.normal_ordered(of.commutator(two_body_hamiltonian, symmetry_1))

1.0*g_{1212} [1^ 0^ 3 2] +
1.0*g_{1222} [2^ 1^ 0^ 3 2 0] +
-1.0*g_{1222} [2^ 1^ 3 2] +
g_{1222} [3^ 0^ 3 2] +
1.0*g_{1222} [3^ 1^ 0^ 3 2 1] +
-g_{1222} [3^ 2^ 0^ 2 1 0] +
-1.0*g_{1212} [3^ 2^ 1 0] +
-1.0*g_{1222} [3^ 2^ 1^ 3 1 0] +
1.0*g_{1222} [3^ 2^ 2 1] +
-g_{1222} [3^ 2^ 3 0]

In [132]:
of.normal_ordered(of.commutator(two_body_hamiltonian, symmetry_2))

g_{1112} [1^ 0^ 2 1] +
-1.0*g_{1112} [1^ 0^ 3 0] +
-2.0*g_{1112} [2^ 1^ 0^ 3 2 0] +
-g_{1112} [2^ 1^ 1 0] +
-g_{1222} [2^ 1^ 3 2] +
1.0*g_{1112} [3^ 0^ 1 0] +
1.0*g_{1222} [3^ 0^ 3 2] +
-2.0*g_{1112} [3^ 1^ 0^ 3 2 1] +
2.0*g_{1112} [3^ 2^ 0^ 2 1 0] +
2.0*g_{1112} [3^ 2^ 1^ 3 1 0] +
1.0*g_{1222} [3^ 2^ 2 1] +
-1.0*g_{1222} [3^ 2^ 3 0]

In [106]:
of.normal_ordered(of.commutator(two_body_hamiltonian, 
                                of.FermionOperator(number_terms[0], 1)
                                * of.FermionOperator(number_terms[1], 1)))

-g_{1112} [1^ 0^ 2 1] +
g_{1112} [1^ 0^ 3 0] +
g_{1122} [1^ 0^ 3 2] +
-g_{1222} [2^ 1^ 0^ 3 2 0] +
g_{1112} [2^ 1^ 1 0] +
-g_{1112} [3^ 0^ 1 0] +
-g_{1222} [3^ 1^ 0^ 3 2 1] +
g_{1222} [3^ 2^ 0^ 2 1 0] +
-g_{1122} [3^ 2^ 1 0] +
g_{1222} [3^ 2^ 1^ 3 1 0]

In [59]:
of.normal_ordered(of.commutator(full_hamiltonian, symmetry_1)).induced_norm()

(4*Abs(a_1*g_{1222}) + 2*Abs(a_1*h_{12}) + 4*Abs(b_1*g_{1222}) + 2*Abs(b_1*h_{12}) + 8*Abs(c_1*g_{1222}) + 2*Abs(2*a_1*g_{1212} - 2*b_1*g_{1212}) + 2*Abs(2*a_1*g_{1112} + 2*c_1*g_{1112} + c_1*h_{12}) + 2*Abs(2*a_1*g_{1122} + 2*b_1*g_{1122} + 2*c_1*g_{1122}) + 2*Abs(2*b_1*g_{1112} + 2*c_1*g_{1112} + c_1*h_{12}))**1.0

In [62]:
of.normal_ordered(of.commutator(full_hamiltonian, symmetry_2)).induced_norm()

(4*Abs(a_2*g_{1112}) + 2*Abs(a_2*h_{12}) + 4*Abs(b_2*g_{1112}) + 2*Abs(b_2*h_{12}) + 8*Abs(c_2*g_{1112}) + 2*Abs(2*a_2*g_{1212} - 2*b_2*g_{1212}) + 2*Abs(2*a_2*g_{1122} + 2*b_2*g_{1122} + 2*c_2*g_{1122}) + 2*Abs(2*a_2*g_{1222} + 2*c_2*g_{1222} + c_2*h_{12}) + 2*Abs(2*b_2*g_{1222} + 2*c_2*g_{1222} + c_2*h_{12}))**1.0